# 07 Final training on the whole development set

The configurations chosen by cross-validation (`results/chosen_configs.json`) are retrained on the 15 development volumes :

- baseline : LA position prior (mean centroid and box) computed on all development labels;
- random forest : 3 seeds;
- U-Net : 3 seeds ( 25 min each on that config).

Models are saved in `models/`. The test set is not used here.

In [1]:
import json
import sys
from pathlib import Path

import joblib  # saves the random forests to files
import torch
from sklearn.ensemble import RandomForestClassifier

# shared functions of the project (src/unet.py and src/utils.py)
sys.path.insert(0, "../src")
from unet import train
from utils import RESULTS, SPLITS, la_prior, load_case, normalize, sample_training_set

SEEDS = [0, 1, 2]  # 3 trainings with different random seeds, to measure the variability
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)
# settings chosen by cross-validation (notebooks 04, 05 and 06)
configs = json.loads((RESULTS / "chosen_configs.json").read_text())
print(configs)

# all 15 development volumes, with normalised intensities
volumes = []
for c in SPLITS["dev"]:
    x, y, _ = load_case(c)
    volumes.append((normalize(x), y))

{'otsu': {'n_classes': 3, 'radius': 9}, 'random_forest': {'max_depth': 20, 'threshold': 0.9, 'n_trees': 100, 'n_per_class': 5000}, 'unet': {'n_iters': 5000}}


In [2]:
# LA position prior, used by the baseline and by the random forest
centroid, box = la_prior([y for _, y in volumes])
prior = {"centroid": centroid.tolist(),
         "box": [[int(box[0].start), int(box[0].stop)], [int(box[1].start), int(box[1].stop)]]}
(MODEL_DIR / "la_prior.json").write_text(json.dumps(prior, indent=2))
prior

{'centroid': [147.33421289076222, 168.77762879520574, 64.20980183467384],
 'box': [[98, 190], [110, 227]]}

In [3]:
# one random forest per seed (the seed changes the sampled voxels and the trees)
rf_cfg = configs["random_forest"]
for seed in SEEDS:
    X, Y = sample_training_set(volumes, box, rf_cfg["n_per_class"], seed)
    rf = RandomForestClassifier(n_estimators=rf_cfg["n_trees"], max_depth=rf_cfg["max_depth"],
                                n_jobs=-1, random_state=seed).fit(X, Y)
    joblib.dump(rf, MODEL_DIR / f"rf_seed{seed}.joblib", compress=3)
    print(f"random forest seed {seed} saved")

random forest seed 0 saved
random forest seed 1 saved
random forest seed 2 saved


In [4]:
# one U-Net per seed (the seed changes the initial weights, the patches and the augmentations)
for seed in SEEDS:
    model = train(volumes, configs["unet"]["n_iters"], seed=seed)
    torch.save(model.state_dict(), MODEL_DIR / f"unet_seed{seed}.pt")
    print(f"U-Net seed {seed} saved")

U-Net seed 0 saved
U-Net seed 1 saved
U-Net seed 2 saved
